In [ ]:
import pandas as pd
import sqlite3
from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import mean_squared_error, accuracy_score
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb
import numpy as np

# ----------------------------
# Connect to SQLite database
# ----------------------------
conn = sqlite3.connect('crime_data.db')

# ----------------------------
# Load data
# ----------------------------
crime_df = pd.read_sql_query("""
    SELECT crimeID, Month, Longitude, Latitude, LSOA_code, Outcome, WD24CD, Type
    FROM crime
    WHERE substr(Month, 1, 7) BETWEEN '2022-01' AND '2025-01';
""", conn)

prices_df = pd.read_sql_query("""
    SELECT WD23CD, Average_Price
    FROM ward_average_prices;
""", conn)

# ----------------------------
# Preprocessing
# ----------------------------
crime_df['Month'] = pd.to_datetime(crime_df['Month'], format='%Y-%m')
crime_df['Year'] = crime_df['Month'].dt.year

yearly_crimes = (
    crime_df[crime_df['Year'].isin([2022, 2023, 2024])]
    .groupby(['LSOA_code', 'Year'])
    .size()
    .unstack(fill_value=0)
    .rename(columns={2022: 'total_crimes_2022', 2023: 'total_crimes_2023', 2024: 'total_crimes_2024'})
    .reset_index()
)

yearly_crimes['crime_trend'] = yearly_crimes['total_crimes_2024'] - yearly_crimes['total_crimes_2023']
yearly_crimes['crime_ma'] = yearly_crimes[['total_crimes_2022', 'total_crimes_2023', 'total_crimes_2024']].mean(axis=1)

# Mapping of LSOA to ward
latest_mapping = crime_df[['LSOA_code', 'WD24CD']].drop_duplicates()
merged_prices = latest_mapping.merge(prices_df, left_on='WD24CD', right_on='WD23CD', how='left')

features_df = yearly_crimes.merge(merged_prices[['LSOA_code', 'Average_Price']], on='LSOA_code', how='left')

# Get target burglary counts
crime_jan2025 = crime_df[
    (crime_df['Month'] == '2025-01-01') & (crime_df['Type'] == 'Burglary')
].groupby('LSOA_code').size().reset_index(name='target_jan2025')

# Final dataset
model_df = features_df.merge(crime_jan2025, on='LSOA_code', how='left')
model_df['target_jan2025'] = model_df['target_jan2025'].fillna(0)
model_df = model_df.dropna()

# ----------------------------
# Classification Target
# ----------------------------
model_df['has_crime'] = (model_df['target_jan2025'] > 0).astype(int)

# ----------------------------
# Feature Scaling
# ----------------------------
features = ['total_crimes_2024', 'crime_trend', 'Average_Price', 'crime_ma']
scaler = StandardScaler()
model_df[features] = scaler.fit_transform(model_df[features])

X = model_df[features]
y_class = model_df['has_crime']
y_reg = model_df['target_jan2025']

# ----------------------------
# Classification
# ----------------------------
clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(X, y_class)
class_preds = clf.predict(X)

# Evaluate classification
acc = accuracy_score(y_class, class_preds)
print(f"Classification accuracy (has any burglary): {acc:.2f}")

# ----------------------------
# Regression on predicted positives only
# ----------------------------
X_reg = X[class_preds == 1]
y_reg_pos = y_reg[class_preds == 1]

reg = xgb.XGBRegressor(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

reg.fit(X_reg, y_reg_pos)

# Predict final values
model_df['predicted_jan2025'] = 0
model_df.loc[class_preds == 1, 'predicted_jan2025'] = reg.predict(X_reg)

# Round to nearest non-negative integer
model_df['predicted_jan2025'] = model_df['predicted_jan2025'].round().clip(lower=0)


# ----------------------------
# Evaluation
# ----------------------------
rmse = np.sqrt(mean_squared_error(model_df['target_jan2025'], model_df['predicted_jan2025']))
print(f"Final RMSE: {rmse:.2f}")

# Output
print(model_df[['LSOA_code', 'target_jan2025', 'predicted_jan2025']].head())


In [ ]:
import matplotlib.pyplot as plt

# Get feature importances
feature_importances = reg.feature_importances_
feature_names = X.columns

# Print values
for name, importance in zip(feature_names, feature_importances):
    print(f"{name}: {importance:.4f}")

# Optional: Plot
plt.figure(figsize=(6, 4))
plt.barh(feature_names, feature_importances)
plt.xlabel('Importance')
plt.title('Feature Importance (XGBoost)')
plt.tight_layout()
plt.show()
